# 05 · Report figures for the monthly series and the query construction

This notebook draws three figures of the report in English, with the same style as the
other figures:

| Figure | Report | Input file (put it in `data/raw/figures_input/`) |
|---|---|---|
| Monthly Google Trends vs MediaCloud, 2015–2025 | Figure 1 | `gt_monthly.csv`, `mc_monthly.csv` |
| Google Trends signal of Mars Bleu across snowball rounds | Figure 12 | `gt_snowball_rounds_MB.csv` |
| Top bigrams and trigrams of MediaCloud articles, Dry January | Figure 13 | `mc_kgrams_DJ.csv` **or** `mc_stories_DJ.csv` |

**Accepted formats.** The loaders accept both the files of the original collection
notebook and those written by notebook 01 of this repository:

* monthly series: columns `campaign, date, interest` (original Google Trends file),
  `campaign, date, count` (original MediaCloud file), or `code, date, value` (notebook 01).
  The campaign may be a label ("Mars Bleu") or a code ("MB");
* snowball rounds: long format `date, value, round` (notebook 01), or wide format
  `date, <round 1>, <round 2>, ...` with one column per round;
* k-grams: counts `n, kgram, count` (notebook 01), or articles with a `text` column, from
  which bigrams and trigrams are counted here.

Each figure is skipped with a message if its input is missing. Outputs are written to
`results/figures/` and, under the file names used by the report's `.tex`, to
`results/figures/report/` (RGB, 1800 px wide, so that LaTeX compiles quickly).

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "phc").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from PIL import Image

from phc.config import CAMPAIGNS, ORDER, DATA_RAW, FIG_DIR
from phc.collect import kgram_counts

IN_DIR = DATA_RAW / "figures_input"
IN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR = FIG_DIR / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

GT_COLOR, MC_COLOR, MC_BASE_COLOR = "#2a6fb0", "#e08a2c", "#9a9a9a"
SHOW_MC_BASE = False      # True: also draw MediaCloud with the base (non-optimised) query
DROP_BOILERPLATE = False  # False: k-grams counted on the raw texts, as when the query was built
TOP_K = 15

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
LABEL_TO_CODE = {CAMPAIGNS[c].label.lower(): c for c in CAMPAIGNS}


def to_code(x):
    x = str(x).strip()
    return x if x in CAMPAIGNS else LABEL_TO_CODE.get(x.lower())


def save(fig, name, report_name):
    """Save the figure at 200 dpi, plus an RGB copy 1800 px wide under the report's name."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    im = Image.open(path).convert("RGB")
    if im.width > 1800:
        im = im.resize((1800, round(im.height * 1800 / im.width)), Image.LANCZOS)
    im.save(REPORT_DIR / f"{report_name}.png", optimize=True)
    print("saved", path.relative_to(ROOT), "and", (REPORT_DIR / f"{report_name}.png").relative_to(ROOT))


def find(*names):
    for n in names:
        if (IN_DIR / n).exists():
            return IN_DIR / n
    return None


print("Input folder:", IN_DIR.relative_to(ROOT))
print("Files found:", sorted(p.name for p in IN_DIR.glob("*")) or "none")

Input folder: data/raw/figures_input
Files found: none


## 1. Monthly series (report Figure 1)

Each series is divided by its maximum over 2015–2025 and multiplied by 100.

In [2]:
def load_monthly(path):
    """Return a frame with columns code, date, value."""
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    camp = cols.get("code") or cols.get("campaign")
    val = cols.get("value") or cols.get("interest") or cols.get("count")
    if camp is None or val is None or "date" not in cols:
        raise ValueError(f"{path.name}: expected columns campaign/code, date and interest/count/value; got {list(df.columns)}")
    out = pd.DataFrame({"code": df[camp].map(to_code), "date": pd.to_datetime(df[cols["date"]]),
                        "value": pd.to_numeric(df[val], errors="coerce")})
    unknown = df.loc[out["code"].isna(), camp].unique()
    if len(unknown):
        print(f"  {path.name}: campaigns not recognised and ignored: {list(unknown)}")
    return out.dropna(subset=["code"])


gt_path = find("gt_monthly.csv", "google_trends_monthly.csv", "GT_monthly.csv")
mc_path = find("mc_monthly.csv", "mediacloud_opti_kuplets.csv", "MC_monthly.csv")
mc_base_path = find("mc_monthly_base.csv", "mediacloud_monthly.csv")

if gt_path and mc_path:
    gt_m, mc_m = load_monthly(gt_path), load_monthly(mc_path)
    mc_b = load_monthly(mc_base_path) if (SHOW_MC_BASE and mc_base_path) else None
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for ax, c in zip(axes.flat, ORDER):
        g = gt_m[gt_m["code"] == c].set_index("date")["value"].sort_index()
        m = mc_m[mc_m["code"] == c].set_index("date")["value"].sort_index()
        ax.plot(g.index, 100 * g / g.max(), color=GT_COLOR, lw=1.6, label="Google Trends")
        ax.plot(m.index, 100 * m / m.max(), color=MC_COLOR, lw=1.3, ls="--", label="MediaCloud (optimised query)")
        if mc_b is not None:
            b = mc_b[mc_b["code"] == c].set_index("date")["value"].sort_index()
            ax.plot(b.index, 100 * b / b.max(), color=MC_BASE_COLOR, lw=1, ls=":", label="MediaCloud (base query)")
        ax.set(title=CAMPAIGNS[c].label, ylabel="Normalised (max = 100)", ylim=(0, 105))
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[0, 0].legend(fontsize=8)
    fig.suptitle("Monthly Google Trends and MediaCloud series, 2015–2025")
    fig.tight_layout()
    save(fig, "fig01_monthly_gt_mc", "figure2")
else:
    print("Figure 1 skipped: put gt_monthly.csv and mc_monthly.csv in", IN_DIR.relative_to(ROOT))

Figure 1 skipped: put gt_monthly.csv and mc_monthly.csv in data/raw/figures_input


## 2. Snowball rounds of the Google Trends query, Mars Bleu (report Figure 12)

If the curve keeps its shape from one round to the next, the terms added by the snowball
add volume but no new dynamics. The correlation of each round with the last one
quantifies this.

In [3]:
def load_rounds(path):
    """Return a wide frame: index date, one column per round."""
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    if "round" in cols and "value" in cols:
        df[cols["date"]] = pd.to_datetime(df[cols["date"]])
        wide = df.pivot_table(index=cols["date"], columns=cols["round"], values=cols["value"])
        wide.columns = [f"Round {r}" for r in wide.columns]
        return wide
    wide = df.set_index(pd.to_datetime(df[cols["date"]])).drop(columns=cols["date"])
    return wide.apply(pd.to_numeric, errors="coerce")


snow_path = find("gt_snowball_rounds_MB.csv", "snowball_rounds_mars_bleu.csv")
if snow_path:
    wide = load_rounds(snow_path)
    fig, ax = plt.subplots(figsize=(12, 4.2))
    for col, color in zip(wide.columns, ["#3a9a5b", "#c8553d", "#7b3294", "#2a6fb0", "#e08a2c"]):
        ax.plot(wide.index, wide[col], lw=1.3, color=color, label=col)
    ax.set(title="Google Trends signal for Mars Bleu across snowball rounds (France, 2015–2025)",
           ylabel="Relative search volume (0–100)")
    ax.legend(fontsize=9)
    fig.tight_layout()
    save(fig, "fig09_snowball_mars_bleu", "figure15b")
    print("Correlation of each round with the last one:")
    print(wide.corr()[wide.columns[-1]].round(3).to_string())
else:
    print("Figure 12 skipped: put gt_snowball_rounds_MB.csv in", IN_DIR.relative_to(ROOT))

Figure 12 skipped: put gt_snowball_rounds_MB.csv in data/raw/figures_input


## 3. Top k-grams of MediaCloud articles, Dry January (report Figure 13)

Bigrams and trigrams are counted after removing French stop words. With
`DROP_BOILERPLATE = False` (default) the raw texts are used, as when the query was built,
so web boilerplate (cookie banners, newsletter links) remains visible, as discussed in the
report.

In [4]:
kg_path = find("mc_kgrams_DJ.csv")
st_path = find("mc_stories_DJ.csv")
kg = None
if kg_path:
    kg = pd.read_csv(kg_path)
elif st_path:
    texts = pd.read_csv(st_path)["text"].dropna().astype(str).tolist()
    kg = kgram_counts(texts, drop_boilerplate=DROP_BOILERPLATE)
    print(f"k-grams counted on {len(texts)} articles")

if kg is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    for ax, n, title in zip(axes, (2, 3), ("Bigrams", "Trigrams")):
        top = kg[kg["n"] == n].nlargest(TOP_K, "count").iloc[::-1]
        ax.barh(top["kgram"], top["count"], color=GT_COLOR)
        for y, v in enumerate(top["count"]):
            ax.text(v, y, f" {v}", va="center", fontsize=8)
        ax.set(title=title, xlabel="Occurrences")
        ax.grid(axis="y", alpha=0)
    fig.suptitle(f"Top {TOP_K} k-grams in MediaCloud articles — Dry January")
    fig.tight_layout()
    save(fig, "fig10_kgrams_dry_january", "figure16")
else:
    print("Figure 13 skipped: put mc_kgrams_DJ.csv or mc_stories_DJ.csv in", IN_DIR.relative_to(ROOT))

Figure 13 skipped: put mc_kgrams_DJ.csv or mc_stories_DJ.csv in data/raw/figures_input
